# satellite-change-agent: ask about a flood, a wildfire or new construction

This notebook runs the measurement of [satellite-change-agent](https://github.com/deltasata/satellite-change-agent)
on Google's computers. **Nothing is installed on your computer**: the code, the trained networks and the
satellite images are all downloaded to Google's temporary machine, which is deleted when you close the tab.

Run the cells from top to bottom with the ▶ buttons, or with *Runtime → Run all*. The first two cells
prepare the machine (about 3-4 minutes); every question after that takes 1-3 minutes, most of it downloading
free Sentinel-2 images.

The language model of the full system (which holds a conversation) is not used here: you give the place,
the kind of event and the date directly, and you get the same measurement the chat agent would call.

## 1. Prepare the machine (about 2 minutes)

In [ ]:
!git clone -q --depth 1 https://github.com/deltasata/satellite-change-agent.git
%cd satellite-change-agent
!pip install -q -e . "omnicloudmask==1.8.1" 2>&1 | grep -v -i "warning\|already" || true

## 2. Get the trained networks (about 1-2 minutes)

The water, radar-water and burn networks come from this project's
[GitHub Release](https://github.com/deltasata/satellite-change-agent/releases/tag/models-v1) (CC BY 4.0), each
checked against its fingerprint. The cloud mask (OmniCloudMask) comes from its authors' own page. The
construction network is not published (its training labels may not be shared), so construction questions
use the simpler change rule here, and the answer says so.

In [ ]:
!python scripts/get_models.py

## 3. Ask a question

Choose an example, or choose **"my own place"** and fill in the fields below it. Then run the cell.

- **flood**: the date is the day of the flood.
- **wildfire**: give the day of the fire, or the first and last day of the month if you only know the month.
- **construction**: give two periods to compare, ideally the same season of two different years.

In [ ]:
#@title Your question { run: "auto" }
example = "Conselice, Italy: flood, 23 May 2023"  #@param ["Conselice, Italy: flood, 23 May 2023", "Altadena, California: wildfire, January 2025", "Grünheide, Germany: construction, summer 2019 vs summer 2021", "my own place"]
latitude = 44.5063  #@param {type:"number"}
longitude = 11.8276  #@param {type:"number"}
kind = "flood"  #@param ["flood", "wildfire", "construction"]
event_day = "2023-05-23"  #@param {type:"string"}
event_last_day = ""  #@param {type:"string"}
before_period = "2019-06-01 to 2019-08-31"  #@param {type:"string"}
after_period = "2021-06-01 to 2021-08-31"  #@param {type:"string"}
size_km = 5.0  #@param {type:"number"}

EXAMPLES = {
    "Conselice, Italy: flood, 23 May 2023": dict(lat=44.5063, lon=11.8276, kind="flood", event_date="2023-05-23"),
    "Altadena, California: wildfire, January 2025": dict(lat=34.19, lon=-118.13, size_km=4.0, kind="vegetation_loss",
                                                         event_date="2025-01-01", event_end="2025-01-31"),
    "Grünheide, Germany: construction, summer 2019 vs summer 2021": dict(
        lat=52.395, lon=13.795, size_km=3.0, kind="construction", before_start="2019-06-01", before_end="2019-08-31",
        after_start="2021-06-01", after_end="2021-08-31"),
}
if example in EXAMPLES:
    question = EXAMPLES[example]
else:
    k = {"flood": "flood", "wildfire": "vegetation_loss", "construction": "construction"}[kind]
    question = dict(lat=latitude, lon=longitude, kind=k, size_km=size_km)
    if k == "construction":
        (b0, b1), (a0, a1) = (p.split(" to ") for p in (before_period, after_period))
        question.update(before_start=b0.strip(), before_end=b1.strip(), after_start=a0.strip(), after_end=a1.strip())
    else:
        question["event_date"] = event_day.strip()
        if event_last_day.strip():
            question["event_end"] = event_last_day.strip()
print(question)

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from IPython.display import Image, display
from core.measure import measure_change

r = measure_change(**question)
print("ANSWER:", r.answer.upper().replace("_", " "))
print("rule:  ", r.rule or r.reason)
if r.caution:
    print("caution:", r.caution)
print()
for s in r.steps:
    print(" -", s)
if r.figure:
    display(Image(r.figure))

### What you see

- **The answer** comes from a fixed rule on the measured numbers (written out on the "rule" line): for a flood, the
  share of the area that is water after the event but not before must be at least 5 %; for a wildfire or new
  construction, the share of the area that changed must be at least 3 %.
- **The figure**: the before and after images (each the median of its cloud-free views), the pixels the network
  marked (blue: new water; orange: burnt or changed), and the cloud-free share of every satellite pass, with the two
  compared periods shaded.
- **Steps**: how many images were found, how much of the ground was seen clearly, and which networks measured.

How each network was chosen, and the tests it had to pass before it was used:
[docs/METHODS.md](https://github.com/deltasata/satellite-change-agent/blob/main/docs/METHODS.md).
Satellite data: Copernicus Sentinel-2 (ESA), via Element 84 Earth Search; radar: Sentinel-1 via Microsoft Planetary Computer.